In [ ]:
# ============================================================
# MPLUS TIME GRID PIPELINE
# BASELINE EMA + STEPS + HEART RATE
# ============================================================


In [ ]:
# ============================================================
# IMPORTS AND SETTINGS
# ============================================================

from pyprojroot import here 
from pathlib import Path
import sys
import pandas as pd
import numpy as np

MISSING = -999

# ============================================================
# PATHS
# ============================================================

# .here is located as invisible file in the project root working directory
PROJECT_ROOT = Path(here())

# add project root to sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# import local paths from server_config
from server_config import preprocessed_path, redcap_path

# convert server paths to Path objects and set path
passive_path = Path(preprocessed_path) / "backup_passive_recent.feather"
ema_path = Path(preprocessed_path) / "ema_content.pkl"
redcap_path_age_sex = Path(redcap_path) / "redcap_data.pkl"
redcap_path_final = Path(redcap_path) / "sp1_cleaned" / "baseline_raw_20251012.sav"

# define output path
OUTPUT_DIR = Path(preprocessed_path) / "tessa" / "dsem"

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

df_passive = pd.read_feather(passive_path)
df_ema = pd.read_pickle(ema_path)
df_redcap = pd.read_pickle(redcap_path_age_sex)
df_redcap_final = pd.read_spss(redcap_path_final)


### 0. Demographics: Sample size, Age, Sex

not relevant for the timegrid (only for poster) - DELETE THIS SECTION LATER

In [ ]:
#for col in df_redcap_final.columns:
#    print(col)

# severe_disease_type_6 = schilddrüsenfunktionsstörung (lag vor oder liegt noch vor); 'sci_cv_prim', 'severe_disease_type___6'
df_redcap_sample = df_redcap_final[['for_id', 'main_sample', 'site', 'nonstarter', 'dropout', 'scid_cv_prim_cat', 'age',
                                   'gender', 'years_of_education', 'ema_id', 'ema_watch_id', 'mensturation', 'menstruation_latest', 
                                   'severe_disease_type___6', 'madrs_0_total', 'bdi_0_sum', 'bsi_0_sum', 'bsi_0_gsi']]

In [ ]:
# age
#df_redcap_sample['age'].describe()

# sex 
df_redcap_sample['gender_recoded'] = pd.to_numeric(
    df_redcap_sample['gender']
    .replace({
        'männlich': 0,
        'mÃ¤nnlich': 0,
        'weiblich': 1,
        'w': 1,
        'm': 0
    }),
    errors='coerce'
)

female_percent = df_redcap_sample['gender_recoded'].mean() * 100

print(f"Female: {female_percent:.1f}%")

In [ ]:
#import plotly.express as px

# Replace 'your_column' with your column name
#fig = px.histogram(
#    df_redcap_final,
#    x='bsi_0_gsi',
#    nbins=20,
#    title='Histogram'
#)

#fig.show()

In [ ]:
# cut-off check
count_below = (df_redcap_final['bsi_0_gsi'] < 0.56).sum()

print(f"People below 0.56: {count_below}")

# werden diese 21 Personen ausgeschlossen?

In [ ]:
# display all item names
#sorted(df_ema["item"].dropna().unique())

### 1. EMA

In [ ]:
# define affect items
na_cols = ['panas_fear1', 
           'panas_fear2', 
           'panas_guilt1',
           'panas_guilt2', 
           'panas_hostility1', 
           'panas_hostility2',
           'panas_sadness1', 
           'panas_sadness2',
           'panas_loneliness']

pa_cols = ['panas_attentiveness', 
           'panas_joviality1', 
           'panas_joviality2',
           'panas_selfassurance',
           'panas_serenity1', 
           'panas_serenity2']

# note: the items ‘panas_shyness’ and ‘panas_fatigue’ have been excluded because they cannot be clearly assigned to either NA or PA

In [ ]:
# ============================================================
# CLEAN EMA
# ============================================================

# 1. keep baseline data only
ema_base = df_ema[df_ema['measurement_burst'] == 0]
# ema_base["measurement_burst"].value_counts() 

# 2. keep relevant columns
df_ema_base = ema_base[['id', 
                        'timestamp_item_completion',
                        'timestamp_beep_completion', 
                        'measurement_burst', 
                        'response', 
                        'item', 
                        'beep_per_person_id',
                        'n_beeps_beeps_completed_per_burst',    # beep count burst (1 - 112)
                        'nr_beep_daily',                        # beep count daily (1 - 8)
                        'n_beeps_completed_per_day',            # sum of beeps completed (per person) within a day
                        'relative_beep_counter'                 # nth beep completed (per person) per burst
                        ]].copy()

# 3. data format 
df_ema_base['id'] = df_ema_base['id'].astype(str)
df_ema_base['response'] = pd.to_numeric(df_ema_base['response'], errors='coerce')

# 4. drop all rows with NaN (prerequisite for pivot)
df_ema_base = df_ema_base.dropna(subset=["id", "beep_per_person_id", "item", "response",])


In [ ]:
# 5. keep affect items only                          TO DO: ADD STRESS ITEMS er_intensity, er_control, event_general
affect_items = pa_cols + na_cols

df_ema_affect_long = df_ema_base[
    df_ema_base["item"].isin(affect_items)
].copy()

df_ema_affect_long.shape

# check item counts
#df_ema_affect_long["item"].value_counts()


In [ ]:
df_ema_affect_long.duplicated(
    subset=["id", "beep_per_person_id", "item"]
).any()

In [ ]:
# 6. pivot ema from long to wide format (this creates one row per person beep)
df_ema_wide = (
    df_ema_affect_long
    .pivot_table(
        index=[
            "id",
            "beep_per_person_id",
        ],
        columns="item",
        values="response",
        aggfunc="first"
    )
    .reset_index()
)

df_ema_wide.head()

In [ ]:
# 7. add beep-level timing and metadata
df_beep_info = (
    df_ema_base
    .groupby(["id", "beep_per_person_id"], as_index=False)
    .agg(
        timestamp_beep_completion=("timestamp_beep_completion", "min"),
        timestamp_item_completion=("timestamp_item_completion", "min"),
        measurement_burst=("measurement_burst", "first"),
        n_beeps_completed_per_burst=("n_beeps_beeps_completed_per_burst", "first"),
        nr_beep_daily=("nr_beep_daily", "first"),
        n_beeps_completed_per_day=("n_beeps_completed_per_day", "first"),
        relative_beep_counter=("relative_beep_counter", "first"),
    )
)

df_ema_wide = df_beep_info.merge(
    df_ema_wide,
    on=["id", "beep_per_person_id"],
    how="left"
)

df_ema_wide.head()

In [ ]:
# 8. create PA and NA scores 
df_ema_wide["pa"] = df_ema_wide[pa_cols].mean(axis=1, skipna=True)
df_ema_wide["na"] = df_ema_wide[na_cols].mean(axis=1, skipna=True)

# descriptive stats
df_ema_wide[["pa", "na"]].describe()


In [ ]:
df_ema_wide.head(10)

### Visualization Affect

#### 1.1 Negative Affect Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# variable
value_col = 'na'

# data
dat = df_ema_wide[value_col].dropna()
mean_na = dat.mean()

# plot
fig, ax = plt.subplots(figsize=(7, 5))

# histogram
sns.histplot(dat, bins=np.arange(0.5, 7.6, 0.5), color='#B7D0C8', edgecolor='#3D6F75', linewidth=0.9, alpha=1, ax=ax)

# mean line
ax.axvline(mean_na, color='#1F5C67', linestyle='--', linewidth=2)

# mean label
ax.text(mean_na + 0.1, ax.get_ylim()[1] * 0.95, f'Overall Mean = {mean_na:.2f}',fontsize=11)

# labels
ax.set_xlabel('Negative Affect (1–7)',fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)

# limits
ax.set_xlim(0, 7)

# publication-style axes
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# move axes slightly outward
ax.spines['left'].set_position(('outward', 10))
ax.spines['bottom'].set_position(('outward', 10))

# thicker axes
ax.spines['left'].set_linewidth(2)
ax.spines['bottom'].set_linewidth(2)

# tick styling
ax.tick_params(axis='both', direction='out', length=5,width=2,labelsize=11)

# no grid
ax.grid(False)

plt.tight_layout()
plt.savefig("negative_affect_histogram.svg", format="svg", bbox_inches="tight")
plt.show()


#### 1.2 Negative Affect Time series plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

id_col = 'id'
value_col = 'na'
time_col = 'timestamp_beep_completion'  

dat = df_ema_wide[[id_col, time_col, value_col]].dropna().copy()
dat[time_col] = pd.to_datetime(dat[time_col])

# define study day per person
dat['date'] = dat[time_col].dt.date

# find IDs with at least 16 days of EMA data
valid_ids = (dat.groupby(id_col)['date'].nunique().loc[lambda x: x >= 16].index)

# choose three random valid IDs from the sample
np.random.seed(123)
selected_ids = np.random.choice(valid_ids, size=3, replace=False)

plot_dat = dat[dat[id_col].isin(selected_ids)].copy()

# create relative study day per ID: day 1, day 2, ..., day 16
plot_dat['study_day'] = (
    plot_dat
    .groupby(id_col)['date']
    .transform(lambda x: pd.factorize(x)[0] + 1)
)

# keep same 16-day window for everyone
plot_dat = plot_dat[plot_dat['study_day'].between(1, 16)]

# daily mean per ID
daily_mean = (
    plot_dat
    .groupby([id_col, 'study_day'], as_index=False)[value_col]
    .mean()
)

# plot
fig, ax = plt.subplots(figsize=(10, 5))

# config plot
palette = sns.color_palette("Set2", n_colors=len(selected_ids))
legend_labels = ['A', 'B', 'C']

for i, pid in enumerate(selected_ids):
    person_raw = plot_dat[plot_dat[id_col] == pid]
    person_mean = daily_mean[daily_mean[id_col] == pid]

    # raw EMA fluctuations
    ax.plot(person_raw['study_day'], person_raw[value_col], color=palette[i], alpha=0.25, linewidth=1)
    ax.scatter(person_raw['study_day'], person_raw[value_col], color=palette[i], alpha=0.25, s=20)

    # daily mean line
    ax.plot(person_mean['study_day'], person_mean[value_col], color=palette[i], linewidth=3, marker='o', 
            label=legend_labels[i])   # A, B, C instead of actual IDs (label=f'ID {pid}')

ax.set_title('EMA trajectories of negative affect across 16 days for three patients', pad=15)

ax.set_xlabel('Day')
ax.set_ylabel('Negative Affect')
ax.set_xlim(1, 16)
ax.set_xticks(range(1, 17))
ax.set_ylim(0, 7)

ax.legend(title='Patient')

sns.despine()
plt.tight_layout()
plt.savefig("negative_affect_timeseries.svg", format="svg", bbox_inches="tight")
plt.show()

#### 2. Positive Affect Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# variable
value_col = 'pa'

# data
dat = df_ema_wide[value_col].dropna()
mean_pa = dat.mean()

# plot
fig, ax = plt.subplots(figsize=(7, 5))

# histogram
sns.histplot(dat, bins=np.arange(0.5, 7.6, 0.5), color='#B7D0C8', edgecolor='#3D6F75', linewidth=0.9, alpha=1, ax=ax)

# mean line
ax.axvline(mean_pa, color='#1F5C67', linestyle='--', linewidth=2)

# mean label
ax.text(mean_pa + 0.1, ax.get_ylim()[1] * 0.97, f'Overall Mean = {mean_pa:.2f}',fontsize=11)

# labels
ax.set_xlabel('Positive Affect (1–7)',fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)

# limits
ax.set_xlim(0, 7)

# publication-style axes
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# move axes slightly outward
ax.spines['left'].set_position(('outward', 10))
ax.spines['bottom'].set_position(('outward', 10))

# thicker axes
ax.spines['left'].set_linewidth(2)
ax.spines['bottom'].set_linewidth(2)

# tick styling
ax.tick_params(axis='both', direction='out', length=5,width=2,labelsize=11)

# no grid
ax.grid(False)

plt.tight_layout()
plt.savefig("positive_affect_histogram.svg", format="svg", bbox_inches="tight")
plt.show()


In [ ]:
# check missingness
df_ema_wide[["pa", "na"]].isna().mean()

In [ ]:
# ============================================================
# CREATE BASELINE TIME GRID FROM THOSE EMA ROWS
# ============================================================

# rename for readability
df_ema_wide = df_ema_wide.rename(
    columns={
        "timestamp_item_completion": "timestamp"
    }
)

# make sure timestamp is datetime
df_ema_wide["timestamp"] = pd.to_datetime(
    df_ema_wide["timestamp"],
    errors="coerce"
)

# sort by person and time
df_ema_wide = df_ema_wide.sort_values(
    ["id", "timestamp"]
).copy()

# first beep per person
df_ema_wide["t0"] = (df_ema_wide.groupby("id")["timestamp"].transform("first"))

# hours since first beep (1 hour = 3600 sec)
df_ema_wide["time_hr"] = ((df_ema_wide["timestamp"] - df_ema_wide["t0"]).dt.total_seconds()/ 3600)

# inspect
df_ema_wide[["id", "timestamp", "time_hr", "pa","na"]].head(10)

# 'time_hr' is the relevant Mplus variable for TINTERVAL = time (2.5)
# if you only want to create a time grid for ema data you can stop here and export it!


In [ ]:
# no missings - CHECK :)
rows_missing_time = df_ema_wide[
    df_ema_wide["timestamp"].isna()
].copy()

rows_missing_time.head()

In [ ]:
# version 2 (v2) 'passive-data windows'
# blocks are always exactly 2.5h long, with the beep being the end of each block

# ensure timestamp is datetime
df_ema_wide["timestamp"] = pd.to_datetime(df_ema_wide["timestamp"])

# sort
df_ema_wide = df_ema_wide.sort_values(["id", "timestamp"]).copy()

# fixed window length: always 2.5h ending at the beep
win = pd.Timedelta(hours=2.5)

# block_end = current EMA timestamp
df_ema_wide["block_end"] = df_ema_wide["timestamp"]

# block_start = ALWAYS 2.5 hours before block_end  (this fixes the long-block problem)
df_ema_wide["block_start"] = df_ema_wide["block_end"] - win

# unique block ID for mapping passive data
df_ema_wide["unique_block"] = (
    df_ema_wide["id"].astype(str)
    + "_"
    + df_ema_wide.groupby("id").cumcount().add(1).astype(str)
)

# create grid object for later mapping
df_grid = df_ema_wide[["id", "unique_block", "timestamp", "block_start", "block_end", "time_hr", "pa", "na"]].copy()

# sanity check: should be all 2.5 hours
(df_grid["block_end"] - df_grid["block_start"]).value_counts().head(5)


In [ ]:
df_grid.head()

In [ ]:
# version 1 (v1) 'passive-data window' INCORRECT, not used
# explanation: blocks run from previous beep → current beep (so block length equals the actual gap between beeps). 
# This is where the “error” comes from: if someone misses beeps, the gap can be days, so you accidentally 
# create multi-day blocks.


# create passive-data windows
# block_end = current EMA timestamp
#df_ema_wide["block_end"] = df_ema_wide["timestamp"]

# block_start = previous EMA timestamp
#df_ema_wide["block_start"] = (df_ema_wide.groupby("id")["timestamp"].shift(1))

# for first beep, use 2.5 hours before first EMA
#df_ema_wide["block_start"] = df_ema_wide["block_start"].fillna(df_ema_wide["block_end"] - pd.Timedelta(hours=2.5))

# unique block ID for mapping passive data
#df_ema_wide["unique_block"] = (df_ema_wide["id"].astype(str)+ "_"+ df_ema_wide.groupby("id").cumcount().add(1).astype(str))

# create grid object for later mapping
#df_grid = df_ema_wide[["id", "unique_block", "timestamp", "block_start", "block_end", "time_hr", "pa", "na",]].copy()

# inspect
#df_grid.head(10)

This time grid is ready for both:
1. Mplus EMA-only .dat export
2. steps / heart rate mapping

### 2. Passive

In [ ]:
# initial inspection
df_passive.head()

In [ ]:
# ============================================================
# CLEAN STEPS 
# ============================================================

df_steps = df_passive[df_passive["modality"] == "Steps"].copy()

df_steps["id"] = df_steps["id"].astype(str)

df_steps["timestamp_start"] = pd.to_datetime(
    df_steps["timestamp_start"],
    errors="coerce"
)

df_steps["timestamp_end"] = pd.to_datetime(
    df_steps["timestamp_end"],
    errors="coerce"
)

# transform step counts to numeric values
df_steps["float_value"] = pd.to_numeric(
    df_steps["float_value"],
    errors="coerce"
)

# remove rows with missing IDs, timestamps, or step counts
df_steps = df_steps.dropna(
    subset=["id", "timestamp_start", "timestamp_end", "float_value"]
)

# exclude negative step counts
df_steps = df_steps[df_steps["float_value"] >= 0]

# calculate duration of each detected walking event in minutes
df_steps["duration_min"] = (
    df_steps["timestamp_end"] - df_steps["timestamp_start"]
).dt.total_seconds() / 60

# remove events with zero or negative duration
df_steps = df_steps[df_steps["duration_min"] > 0]

# compute walking rate (steps per minute)
df_steps["steps_per_min"] = df_steps["float_value"] / df_steps["duration_min"]

# in line with Hammelrath et al. (2026): 200 steps per minute as threshold, remove implausible rate (> 200 steps/min)
# threshold based on reported upper limits of human walking rate
df_steps = df_steps[df_steps["steps_per_min"] <= 200] 

# remove temporary variables used for quality control
df_steps = df_steps.drop(columns=["duration_min", "steps_per_min"])

# inspect first 100 cleaned records
df_steps.head(100)


In [ ]:
# ============================================================
# CLEAN HR 
# ============================================================

df_hr = df_passive[df_passive["modality"] == "HeartRate"].copy()

df_hr["id"] = df_hr["id"].astype(str)

df_hr["timestamp_start"] = pd.to_datetime(
    df_hr["timestamp_start"],
    errors="coerce"
)

df_hr["float_value"] = pd.to_numeric(
    df_hr["float_value"],
    errors="coerce"
)

df_hr = df_hr.dropna(
    subset=["id", "timestamp_start", "float_value"]
)

# in line with Hammelrath et al. (2026) p. 3
df_hr = df_hr[df_hr["float_value"].between(30, 220)]

df_hr.head()

In [ ]:
# ============================================================
# RESTRICT PASSIVE DATA TO BASELINE
# ============================================================

baseline_period = (
    df_grid
    .groupby("id")
    .agg(
        baseline_start=("block_start", "min"),
        baseline_end=("block_end", "max")
    )
    .reset_index()
)

baseline_period.head()

In [ ]:
# STEPS
df_steps = df_steps.merge(
    baseline_period,
    on="id",
    how="inner"
)

df_steps = df_steps[
    (df_steps["timestamp_start"] <= df_steps["baseline_end"]) &
    (df_steps["timestamp_end"] >= df_steps["baseline_start"])
].copy()

df_steps = df_steps.drop(columns=["baseline_start", "baseline_end"])

df_steps.shape

In [ ]:
# HEART RATE 
df_hr = df_hr.merge(
    baseline_period,
    on="id",
    how="inner"
)

df_hr = df_hr[
    (df_hr["timestamp_start"] >= df_hr["baseline_start"]) &
    (df_hr["timestamp_start"] <= df_hr["baseline_end"])
].copy()

df_hr = df_hr.drop(columns=["baseline_start", "baseline_end"])

df_hr.shape


In [ ]:
# ============================================================
# MAP PASSIVE DATA INTO BASELINE GRID
# ============================================================

# map steps to ema blocks
joined_steps = df_steps.merge(
    df_grid[
        [
            "id",
            "unique_block",
            "block_start",
            "block_end"
        ]
    ],
    on="id",
    how="inner"
)

joined_steps = joined_steps[
    (joined_steps["timestamp_start"] < joined_steps["block_end"]) &
    (joined_steps["timestamp_end"] > joined_steps["block_start"])
].copy()

joined_steps.shape

In [ ]:
# calculate weighted steps

# For each step record and EMA block, take the later start time 
# (the overlap cannot start before the EMA block starts)
joined_steps["overlap_start"] = joined_steps[
    ["timestamp_start", "block_start"]
].max(axis=1)

# For each step record and EMA block, take the earlier end time 
# (the overlap cannot continue after the step record ends)
joined_steps["overlap_end"] = joined_steps[
    ["timestamp_end", "block_end"]
].min(axis=1)

# calculate how many seconds the step record overlaps with the EMA block
joined_steps["overlap_seconds"] = (
    joined_steps["overlap_end"] - joined_steps["overlap_start"]
).dt.total_seconds()

# calculate the total duration of the original step record
joined_steps["sensor_seconds"] = (
    joined_steps["timestamp_end"] - joined_steps["timestamp_start"]
).dt.total_seconds()

# remove invalid step records where the duration is zero or negative
joined_steps = joined_steps[joined_steps["sensor_seconds"] > 0]

# assign only the proportional amount of steps that belongs to the EMA block
# example: overlap_seconds = 1800, sensor_seconds  = 3600, float_value = 600
# weighted_steps = 1800 / 3600 × 600 = 300 -> only 300 of the 600 steps are assigned to this EMA block
joined_steps["weighted_steps"] = (
    joined_steps["overlap_seconds"] / joined_steps["sensor_seconds"]
) * joined_steps["float_value"]

# sum all weighted step records within each EMA block
steps_summary = (
    joined_steps
    .groupby("unique_block")["weighted_steps"]
    .sum()
    .reset_index(name="steps")
)

steps_summary.head()

In [ ]:
# merge onto grid
df_grid = df_grid.merge(
    steps_summary,
    on="unique_block",
    how="left"
)

df_grid.head()

In [ ]:
# steps are highly skewed
#df_grid["steps"].hist(bins=50)

# if it looks strongly skewed use log transformation to reduce skewness and stabilize estimation
#np.log1p()

#raw steps: absolute physical activity volume
#log-transformed steps: relative/intensity-scaled activity variation 

In [ ]:
# map heart rate to ema blocks
joined_hr = df_hr.merge(
    df_grid[
        [
            "id",
            "unique_block",
            "block_start",
            "block_end"
        ]
    ],
    on="id",
    how="inner"
)

joined_hr = joined_hr[
    (joined_hr["timestamp_start"] >= joined_hr["block_start"]) &
    (joined_hr["timestamp_start"] < joined_hr["block_end"])
].copy()

joined_hr.shape


In [ ]:
# aggregate HR
hr_summary = (
    joined_hr
    .groupby("unique_block")["float_value"]
    .agg(
        hr_mean="mean",
        hr_median="median",
        hr_sd="std",   
        hr_n="count"
    )
    .reset_index()
)

hr_summary.head()

In [ ]:
# merge into grid
df_grid = df_grid.merge(
    hr_summary,
    on="unique_block",
    how="left"
)

df_grid.head()

In [ ]:
# check final missingness
df_grid[["pa", "na", "steps", "hr_mean"]].isna().mean()

In [ ]:
# check distributions
df_grid[["pa", "na", "steps", "hr_mean"]].describe()

In [ ]:
df_grid[["pa", "na", "steps", "hr_mean"]].median()

### Visualization weighted step count and heart rate

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_distribution(df, value_col, xlabel, filename=None, percentile=None):
    x = df[value_col].dropna()

    mean_x = x.mean()
    median_x = x.median() if value_col == 'steps' else None

    if percentile is not None:
        cutoff = x.quantile(percentile)
        x = x[x <= cutoff]

    fig, ax = plt.subplots(figsize=(7, 5))

    sns.histplot(x, bins=40, color='#B7D0C8', edgecolor='#3D6F75', linewidth=0.9, alpha=1, ax=ax)

    # mean line
    ax.axvline(mean_x, color='#1F5C67', linestyle='--', linewidth=2)
    ax.text(mean_x + (ax.get_xlim()[1] - ax.get_xlim()[0]) * 0.02, ax.get_ylim()[1] * 0.95, f'Overall Mean = {mean_x:.2f}', fontsize=11)

    # median line (only for steps)
    if value_col == 'steps':ax.axvline(median_x, color='#1F5C67', linestyle='--', linewidth=2)
    if value_col == 'steps':ax.text(median_x + (ax.get_xlim()[1] - ax.get_xlim()[0]) * 0.001, ax.get_ylim()[1] * 1.02, f'Overall Median = {median_x:.2f}', fontsize=11)

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.spines['left'].set_position(('outward', 10))
    ax.spines['bottom'].set_position(('outward', 10))

    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)

    ax.tick_params(
        axis='both',
        direction='out',
        width=2,
        length=5,
        labelsize=11
    )

    ax.grid(False)

    plt.tight_layout()

    if filename is not None:
        plt.savefig(filename, format="svg", bbox_inches="tight")

    plt.show()

In [ ]:
plot_distribution(
    df_grid,
    value_col='steps',
    xlabel='Weighted Steps per EMA Interval',
    filename='weighted_steps_distribution.svg',
    percentile=0.99
)

In [ ]:
# inspect the 20 highest weighted step counts
df_grid[['steps']].sort_values('steps', ascending=False).head(20)

# indices of the 20 largest step count rows
top_idx = df_grid['steps'].nlargest(20).index

# show 20 largest step count rows
filtered = df_grid.loc[top_idx]   # all columns for those rows
#filtered.head(20)

# sanity check: looks fine, just activity spiking but within the determined time window 

In [ ]:
# sanity check: inspect range and percentile of the activity spikes 

#x = df_grid['steps'].dropna()
#x.quantile([.90, .95, .99, .995, .999, ])

In [ ]:
plot_distribution(
    df_grid,
    value_col='hr_mean',
    xlabel='Mean Heart Rate per EMA Interval',
    filename='heart_rate_distribution.svg'
)

In [ ]:
# optional: log-transform steps (recommended for modeling?) 
df_grid["steps_log"] = np.log1p(df_grid["steps"])

df_grid[["steps", "steps_log"]].describe()

df_grid.head()

In [ ]:
# log transformatiom because the step count is highly right-skewed and contains occasional high-activity bursts: 
# this reduces skewness and makes the distribution more symmetric, 
# compresses extreme values, preventing a few highly active periods from dominating model estimation, stabilizes variance
# allows effects to be interpreted as changes associated with proportional rather than absolute differences in activity

df_grid[['steps', 'steps_log']].describe()

In [ ]:
# ============================================================
# CREATE FINAL MPLUS DATASET
# ============================================================

mplus_affect_steps_hr = df_grid[
    [
        "id",
        "time_hr",
        "pa",
        "na",
        "steps",
        "steps_log",
        "hr_mean",
        "hr_median",
        "hr_sd"
    ]
].copy()


# replace infinite values
mplus_affect_steps_hr = mplus_affect_steps_hr.replace(
    [np.inf, -np.inf],
    np.nan
)

# replace missing values for Mplus (cannot handle NaN, only -999)
mplus_affect_steps_hr = mplus_affect_steps_hr.fillna(MISSING)

# important for later modeling: ensure appropriate sorting (in ascending order) -> observations must be ordered
mplus_affect_steps_hr = (mplus_affect_steps_hr.sort_values(["id", "time_hr"]).reset_index(drop=True))

# sanity check
mplus_affect_steps_hr.head(20)

In [ ]:
#stable numeric ids after sorting by original id
mplus_affect_steps_hr["id_num"] = (
    pd.factorize(mplus_affect_steps_hr["id"], sort=True)[0] + 1
)

# optional: save mapping for later merge-back
id_map = (mplus_affect_steps_hr[["id", "id_num"]]
          .drop_duplicates()
          .sort_values("id_num"))
id_map.to_csv(OUTPUT_DIR / "id_mapping.csv", index=False)


mplus_affect_steps_hr.head()

In [ ]:
# save .csv file (with headers) / good for inspection
mplus_affect_steps_hr.to_csv(
    OUTPUT_DIR / "mplus_affect_steps_hr_v2.csv",
    index=False
)

# save .dat file for Mplus (space-delimited, header-less)
# IMPORTANT: id_num must be the FIRST column because your Mplus NAMES order assumes that
mplus_cols = [
    "id_num", "time_hr", "pa", "na", "steps_log", "steps", "hr_mean", "hr_median", "hr_sd"
]

# save .dat file for Mplus (tab-delimited, header-less: suitable for Mplus)
mplus_affect_steps_hr[mplus_cols].to_csv(
    OUTPUT_DIR / "mplus_affect_steps_hr_v2.dat",
    sep=" ",
    header=False,
    index=False
)

In [ ]:
# example reponse distribution
#df_ema_wide['panas_attentiveness'].value_counts().sort_index()